# Gold: das Sternschema für dynamische Abfragen

**Zweck:** Dieses Notebook belegt zwei Entscheidungen der Gold-Schicht und
rechnet beide nach.

1. **Warum Gold nicht aggregiert.** Der ursprüngliche Entwurf sah zwei
   vorberechnete Würfel vor. Gemessen bringen sie nichts — und kosten genau
   die Beweglichkeit, für die die Auswertungsplattform da ist.
2. **Wie Tracts einer Stadt zugeordnet werden.** Das ist der eigentliche Grund
   für diese Schicht: Ohne Stadtgrenze gibt es keinen Nenner, und damit keine
   Rate je Einwohner.

**Stand:** 12. September 2026

In [ ]:
import sys
import time
from pathlib import Path

import duckdb

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'config').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'config').exists():
    raise RuntimeError('Projekt-Root mit config/ nicht gefunden')
if str(PROJECT_ROOT / 'src' / 'crime') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src' / 'crime'))

con = duckdb.connect()
con.execute("SET TimeZone='UTC'")

GOLD = ('fact_offense', 'dim_tract', 'dim_offense', 'dim_date', 'dim_city_day')
fehlend = []
for name in GOLD:
    pfad = PROJECT_ROOT / 'data' / 'gold' / f'dataset={name}' / 'part-0001.parquet'
    if pfad.exists():
        con.execute(f"CREATE VIEW {name} AS SELECT * FROM read_parquet('{pfad}')")
    else:
        fehlend.append(name)

_silber = PROJECT_ROOT / 'data' / 'silver' / 'dataset=crime_offense' / 'part-0001.parquet'
SILBER = f"read_parquet('{_silber}')" if _silber.exists() else None

if fehlend:
    print('Gold fehlt:', fehlend)
    print('Zuerst:  python -m gold_pipeline.build_gold')
else:
    print(f"{'Tabelle':16}{'Zeilen':>12}")
    for name in GOLD:
        n = con.execute(f'SELECT count(*) FROM {name}').fetchone()[0]
        print(f'{name:16}{n:>12,}')


def miss(sql, laeufe=5):
    """Kleinste gemessene Laufzeit in Millisekunden, plus Zeilenzahl."""
    con.execute(sql).fetchall()
    zeiten = []
    for _ in range(laeufe):
        start = time.perf_counter()
        zeilen = con.execute(sql).fetchall()
        zeiten.append((time.perf_counter() - start) * 1000)
    return min(zeiten), len(zeilen)

## 1. Die Messung, die den Entwurf geändert hat

Geplant waren `fact_tract_month` (`geoid` × Monat × Deliktfamilie) und
`fact_city_day` (Stadt × Tag × Familie). Die Begründung war Geschwindigkeit.

Die nächste Zelle misst dieselben Fragen **direkt auf der Silver-Tabelle** —
ohne jede Vorberechnung, ohne Index, ohne Würfel.

In [ ]:
if SILBER:
    fragen = [
        ('ein Delikt, ein Tag, eine Stadt', f"""
            SELECT count(*) FROM {SILBER}
            WHERE city='chicago' AND nibrs_code='23C'
              AND occurred_date_local=DATE '2024-11-09'"""),
        ('ein Delikt, ein Monat, beide Staedte', f"""
            SELECT city, count(*) FROM {SILBER}
            WHERE nibrs_code='240'
              AND date_trunc('month',occurred_date_local)=DATE '2024-06-01'
            GROUP BY 1"""),
        ('eine Deliktfamilie, je Monat und Stadt', f"""
            SELECT city, date_trunc('month',occurred_date_local) m, count(*)
            FROM {SILBER} WHERE nibrs_code LIKE '13%' GROUP BY 1,2"""),
        ('eine Stadt, alle Delikte, je Tag', f"""
            SELECT occurred_date_local, count(*) FROM {SILBER}
            WHERE city='detroit' GROUP BY 1"""),
        ('frei: Tract x Deliktfamilie', f"""
            SELECT tract_geoid, left(nibrs_code,2), count(*) FROM {SILBER}
            WHERE tract_geoid IS NOT NULL GROUP BY 1,2"""),
        ('die teuerste: alles x alles', f"""
            SELECT city, tract_geoid, occurred_date_local, nibrs_code, count(*)
            FROM {SILBER} GROUP BY 1,2,3,4"""),
    ]
    print(f"{'Abfrage auf Silver, ohne Vorberechnung':44}{'ms':>8}{'Zeilen':>10}")
    for name, sql in fragen:
        ms, zeilen = miss(sql)
        print(f'{name:44}{ms:>8.1f}{zeilen:>10,}')

**Der Engpass ist nicht die Datenmenge.** Jede realistische Frage liegt im
einstelligen Millisekundenbereich; selbst die pathologische Gruppierung über
alle vier Achsen bleibt unter einer Zehntelsekunde.

Die nächste Zelle prüft die Gegenrichtung: Spart eine **vorberechnete
Monatsspalte** überhaupt etwas?

In [ ]:
if SILBER:
    con.execute(f"""CREATE OR REPLACE TABLE mit_monat AS
                    SELECT *, date_trunc('month', occurred_date_local) AS monat
                    FROM {SILBER}""")
    a, _ = miss(f"""SELECT city, date_trunc('month',occurred_date_local), nibrs_code,
                    count(*) FROM {SILBER} GROUP BY 1,2,3""")
    b, _ = miss("""SELECT city, monat, nibrs_code, count(*)
                   FROM mit_monat GROUP BY 1,2,3""")
    print(f'date_trunc zur Laufzeit : {a:6.1f} ms')
    print(f'Spalte liegt schon da   : {b:6.1f} ms')
    print(f'Ersparnis               : {a - b:6.1f} ms')
    con.execute('DROP TABLE mit_monat')

## 2. Was die Aggregation gekostet hätte

Die Zeitersparnis ist also keine. Der Preis dagegen ist konkret: Ein Würfel
beantwortet nur die Fragen, auf die er gebaut wurde.

In [ ]:
if SILBER:
    delikte, familien, tage, tracts = con.execute(f"""
        SELECT count(DISTINCT nibrs_code), count(DISTINCT left(nibrs_code,2)),
               count(DISTINCT occurred_date_local), count(DISTINCT tract_geoid)
        FROM {SILBER}""").fetchone()
    print(f'In den Falldaten: {delikte} Delikte, {familien} Familien, '
          f'{tage} Tage, {tracts} Tracts')
    print()
    print(f"{'Tabelle':44}{'Zeilen':>10}")
    for name, sql in (
        ('fact_tract_month (geoid x Monat x Familie)', f"""
            SELECT count(*) FROM (SELECT tract_geoid,
              date_trunc('month',occurred_date_local), left(nibrs_code,2)
              FROM {SILBER} GROUP BY 1,2,3)"""),
        ('fact_city_day (Stadt x Tag x Familie)', f"""
            SELECT count(*) FROM (SELECT city, occurred_date_local,
              left(nibrs_code,2) FROM {SILBER} GROUP BY 1,2,3)"""),
        ('das Korn behalten (eine Zeile je Delikt)', f'SELECT count(*) FROM {SILBER}'),
    ):
        print(f'{name:44}{con.execute(sql).fetchone()[0]:>10,}')

**Drei Fragen, die ein Würfel nicht mehr beantwortet:**

| Frage | woran sie scheitert |
| --- | --- |
| „nur Shoplifting" | beide Würfel fassen die Delikte zu Familien zusammen |
| „der 9. November" | `fact_tract_month` kennt nur Monate |
| „in welchem Tract" | `fact_city_day` kennt nur die Stadt |

Und `fact_tract_month` spart nicht einmal viel Platz — es ist kein Zehntel der
Faktentabelle, sondern gut ein Viertel.

**Gold behält deshalb das Korn** und wird statt dessen *abfragbar* gemacht: eine
Faktentabelle je Delikt, vier kleine Dimensionen daneben.

## 3. Die Stadtzuordnung — der eigentliche Grund für diese Schicht

Bis hierher konnte das Projekt **keine Rate je Einwohner belegen**.
`silver/tract` deckt ganze Countys ab, und ein Tract sagt nicht, zu welcher
Stadt er gehört. Cook County hat 5,18 Millionen Einwohner, Chicago rund 2,7
Millionen — wer den County als Nenner nimmt, halbiert jede Rate.

`silver/place` liefert jetzt die Stadtgrenzen aus TIGER/Line. Damit stellt sich
die eigentliche Frage: **Tracts liegen nicht sauber in Stadtgrenzen.** Sie
überschreiten sie. Drei Regeln wären denkbar, und die nächste Zelle rechnet
alle drei.

In [ ]:
_tract = PROJECT_ROOT / 'data' / 'silver' / 'dataset=tract' / 'vintage=2024' / 'part-0001.parquet'
_place = PROJECT_ROOT / 'data' / 'silver' / 'dataset=place' / 'vintage=2024' / 'part-0001.parquet'

if _tract.exists() and _place.exists():
    con.execute('INSTALL spatial'); con.execute('LOAD spatial')
    from gold_pipeline import STADT_SCHWELLE, STUDIENSTAEDTE   # noqa: E402

    codes = ', '.join(f"('{g}','{s}')" for g, s in sorted(STUDIENSTAEDTE.items()))
    con.execute(f"""CREATE OR REPLACE TABLE anteil AS
      WITH stadt(city_geoid, city) AS (VALUES {codes}),
      grenze AS (
        SELECT s.city, ST_GeomFromWKB(p.geometry) AS geom
        FROM read_parquet('{_place}') p JOIN stadt s ON s.city_geoid = p.geoid
      )
      SELECT t.geoid, t.population, g.city,
             ST_Area(ST_Intersection(ST_GeomFromWKB(t.geometry), g.geom))
               / nullif(ST_Area(ST_GeomFromWKB(t.geometry)),0) AS flaechenanteil
      FROM read_parquet('{_tract}') t
      JOIN grenze g ON ST_Intersects(ST_GeomFromWKB(t.geometry), g.geom)""")

    print('Tracts, die eine Stadtgrenze beruehren:')
    for r in con.execute("""
        SELECT city, count(*) beruehrt,
               count(*) FILTER (WHERE flaechenanteil > 0.999) ganz_drin,
               count(*) FILTER (WHERE flaechenanteil BETWEEN 0.001 AND 0.999) geteilt,
               count(*) FILTER (WHERE flaechenanteil <= 0.001) nur_beruehrt
        FROM anteil GROUP BY 1 ORDER BY 1""").fetchall():
        print(f'  {r[0]:9} beruehrt {r[1]:>4}, ganz drin {r[2]:>4}, '
              f'geteilt {r[3]:>3}, nur beruehrt {r[4]:>3}')

    print()
    print(f"{'Regel':28}{'Chicago':>12}{'Detroit':>12}")
    regeln = [
        ('alle beruehrten Tracts', 'sum(population)'),
        ('Flaechengewichtung', 'round(sum(population*flaechenanteil))'),
        (f'Mehrheitsregel (>{STADT_SCHWELLE})',
         f'sum(population) FILTER (WHERE flaechenanteil > {STADT_SCHWELLE})'),
    ]
    for name, ausdruck in regeln:
        werte = dict(con.execute(
            f'SELECT city, {ausdruck} FROM anteil GROUP BY 1').fetchall())
        print(f"{name:28}{int(werte['chicago']):>12,}{int(werte['detroit']):>12,}")
    print(f"{'Zensus 2020':28}{2_746_388:>12,}{639_111:>12,}")

**Die naive Regel ist unbrauchbar.** „Alle berührten Tracts" zählt Chicago um
16 und Detroit um 38 Prozent zu hoch — sie schluckt jeden Nachbarort, der eine
Grenze streift. Bei Detroit sind das 78 Tracts, die die Stadtgrenze nur
berühren, ohne nennenswert in ihr zu liegen.

**Gewählt ist die Mehrheitsregel:** Ein Tract gehört zu der Stadt, die mehr als
die Hälfte seiner Fläche bedeckt. Zwei Gründe:

- Sie gibt jedem Tract **genau eine** Stadt. Das ist die Voraussetzung dafür,
  dass sich Fälle ohne Doppelzählung summieren lassen — bei einer
  Flächengewichtung müsste jeder Fall anteilig auf zwei Städte verteilt werden,
  was bei Fallzahlen keinen Sinn ergibt.
- Sie liegt praktisch gleichauf mit der Flächengewichtung. Der Unterschied ist
  klein genug, dass die einfachere Regel nichts kostet.

**Detroit trifft die amtliche Zahl auf unter ein Prozent.** Der grössere Abstand
bei Chicago ist keine Ungenauigkeit der Regel, sondern der Unterschied zwischen
einer ACS-Fünfjahresschätzung und einer Zählung.

`dim_tract` hält den Flächenanteil trotzdem in jeder Zeile fest — wer
flächengewichtet rechnen will, kann es, und wer die Regel prüfen will, auch.

### 3.1 Was die Regel an den Rändern kostet

Eine Zuordnung nach Mehrheit heisst: Fälle in Tracts, die *knapp* ausserhalb
liegen, fallen aus dem Nenner. Die nächste Zelle misst, wie viele das sind.

In [ ]:
if 'fact_offense' not in fehlend:
    print('Delikte nach meldender Stadt und Stadt des Tracts:')
    for r in con.execute("""
        SELECT f.reporting_city, coalesce(t.city, '(keine)') AS tract_stadt,
               count(*) AS n, round(avg(t.city_area_share), 3) AS anteil
        FROM fact_offense f JOIN dim_tract t ON t.geoid = f.tract_geoid
        GROUP BY 1, 2 ORDER BY 1, 3 DESC""").fetchall():
        print(f'  meldend {r[0]:9} Tract {r[1]:9} {r[2]:>8,}  '
              f'mittlerer Flaechenanteil {r[3]}')

    gesamt, verortet, ohne_stadt = con.execute("""
        SELECT count(*), count(f.tract_geoid),
               count(*) FILTER (WHERE t.city IS NULL AND f.tract_geoid IS NOT NULL)
        FROM fact_offense f LEFT JOIN dim_tract t ON t.geoid = f.tract_geoid""").fetchone()
    print()
    print(f'Delikte insgesamt          : {gesamt:,}')
    print(f'davon in einem Tract       : {verortet:,} = {verortet/gesamt:.1%}')
    print(f'davon in keiner Studienstadt: {ohne_stadt:,} = {ohne_stadt/gesamt:.1%}')

**Rund 98 Prozent der Fälle liegen in einem Tract der meldenden Stadt.** Der
Rest sitzt an der Grenze — Fälle, die eine Stadtpolizei knapp jenseits der
Stadtgrenze erfasst hat, und Tracts, die zu weniger als der Hälfte in der Stadt
liegen.

Das ist der Preis der Regel, und er ist **gemessen statt geschätzt**. Für eine
Rate je Einwohner ist er vertretbar: Zähler und Nenner folgen derselben Grenze,
also verzerrt der Rand beide Seiten gleichsinnig.

## 4. Die Rate, die es vorher nicht gab

Damit lässt sich zum ersten Mal beantworten, was das Projekt von Anfang an
fragen wollte — und was mit absoluten Fallzahlen nicht zu beantworten war,
weil Chicago rund viermal so viele Einwohner hat wie Detroit.

In [ ]:
if 'fact_offense' not in fehlend:
    for r in con.execute("""
        WITH einwohner AS (
            SELECT city, sum(population) AS n FROM dim_tract
            WHERE city IS NOT NULL GROUP BY 1
        )
        SELECT t.city, count(*) AS delikte, any_value(e.n) AS einwohner,
               round(count(*) * 1000.0 / any_value(e.n), 1) AS je_1000
        FROM fact_offense f
        JOIN dim_tract t ON t.geoid = f.tract_geoid
        JOIN einwohner e ON e.city = t.city
        WHERE t.city IS NOT NULL GROUP BY 1 ORDER BY 1""").fetchall():
        print(f'  {r[0]:9} {r[1]:>8,} Delikte  {r[2]:>10,} Einwohner  '
              f'{r[3]:>6} je 1000')

    print()
    print('Die groessten Deliktfamilien, je 1000 Einwohner:')
    print(f"  {'Familie':9}{'Chicago':>10}{'Detroit':>10}{'Verhaeltnis':>13}")
    for r in con.execute("""
        WITH einwohner AS (
            SELECT city, sum(population) AS n FROM dim_tract
            WHERE city IS NOT NULL GROUP BY 1
        ),
        rate AS (
            SELECT t.city, f.nibrs_family,
                   count(*) * 1000.0 / any_value(e.n) AS je_1000
            FROM fact_offense f
            JOIN dim_tract t ON t.geoid = f.tract_geoid
            JOIN einwohner e ON e.city = t.city
            WHERE t.city IS NOT NULL GROUP BY 1, 2
        )
        SELECT nibrs_family,
               round(max(je_1000) FILTER (WHERE city='chicago'), 1),
               round(max(je_1000) FILTER (WHERE city='detroit'), 1)
        FROM rate GROUP BY 1
        HAVING max(je_1000) FILTER (WHERE city='chicago') > 3
           AND max(je_1000) FILTER (WHERE city='detroit') > 3
        ORDER BY 2 DESC""").fetchall():
        v = f'{r[2]/r[1]:.2f}x' if r[1] else '-'
        print(f'  {r[0]:9}{r[1]:>10}{r[2]:>10}{v:>13}')

**Detroit liegt je Einwohner deutlich höher als Chicago** — und das ist die
Aussage, die absolute Fallzahlen genau umgekehrt nahegelegt hätten, weil
Chicago in jeder Kategorie mehr Fälle zählt.

Zwei Vorbehalte gehören dazu:

- **Anzeigeverhalten ist nicht Kriminalität.** Beide Quellen zählen gemeldete
  Delikte, und die Meldebereitschaft ist zwischen zwei Städten nicht konstant.
- **Der Nenner ist eine ACS-Schätzung**, keine Zählung, und die Stadtzuordnung
  ist die Mehrheitsregel aus Abschnitt 3.

## 5. Die Fragen der Auswertungsplattform, gemessen

Zum Schluss die Probe aufs Ganze: genau die Abfragen, für die das Sternschema
gebaut ist — ein Delikt auswählen, dann Tag, Monat, Jahr; oder nur eine Stadt.

In [ ]:
if 'fact_offense' not in fehlend:
    fragen = [
        ('ein Delikt + ein Tag + eine Stadt', """
            SELECT count(*) FROM fact_offense f JOIN dim_offense o USING (nibrs_code)
            WHERE o.nibrs_description = 'Shoplifting'
              AND f.date = DATE '2024-11-09' AND f.reporting_city = 'chicago'"""),
        ('ein Delikt + ein Monat, beide Staedte', """
            SELECT f.reporting_city, count(*) FROM fact_offense f
            JOIN dim_date d USING (date)
            WHERE f.nibrs_code = '240' AND d.month = 6 GROUP BY 1"""),
        ('nur eine Stadt, Delikte je Monat', """
            SELECT d.month, count(*) FROM fact_offense f JOIN dim_date d USING (date)
            WHERE f.reporting_city = 'detroit' GROUP BY 1 ORDER BY 1"""),
        ('Rate je 1000 Einwohner, Stadt x Delikt', """
            WITH e AS (SELECT city, sum(population) n FROM dim_tract
                       WHERE city IS NOT NULL GROUP BY 1)
            SELECT t.city, f.nibrs_code, count(*) * 1000.0 / any_value(e.n)
            FROM fact_offense f JOIN dim_tract t ON t.geoid = f.tract_geoid
            JOIN e ON e.city = t.city WHERE t.city IS NOT NULL GROUP BY 1, 2"""),
        ('Wochenende gegen Werktag, je Familie', """
            SELECT f.nibrs_family, d.is_weekend, count(*) FROM fact_offense f
            JOIN dim_date d USING (date) GROUP BY 1, 2"""),
        ('warme gegen kalte Tage, eine Stadt', """
            SELECT w.temp_max_c > 25 AS warm, count(*) FROM fact_offense f
            JOIN dim_city_day w ON w.city = f.reporting_city AND w.date = f.date
            WHERE f.reporting_city = 'chicago' GROUP BY 1"""),
    ]
    print(f"{'Abfrage auf Gold':40}{'ms':>8}{'Zeilen':>9}")
    for name, sql in fragen:
        ms, zeilen = miss(sql)
        print(f'{name:40}{ms:>8.1f}{zeilen:>9,}')

**Jede Frage ist ein Filter und ein `GROUP BY` auf einer Tabelle.** Die
Oberfläche baut die `WHERE`-Klausel aus den Auswahlfeldern; mehr braucht es
nicht. Kein Würfel muss vorher wissen, welche Kombination jemand anklickt.

**Und die Dimensionen tragen die Auswahllisten gleich mit:** `dim_offense`
liefert die Delikte und ihre Familien, `dim_date` Monate und Wochentage,
`dim_tract` die Städte. Ein Auswahlfeld ist damit ein `SELECT DISTINCT` auf
eine Dimension und keine gepflegte Liste im Frontend.

## 5.1 Zurückgezogene Codes müssen auflösbar sein

Vier NIBRS-Codes hat das FBI gestrichen; die Fassung 2025 kennt sie nicht mehr.
Die MICR-Quellen führen sie weiter, und **zwei stehen in den Falldaten 2024**.

Sie einfach wegzulassen wäre der schlechteste Weg — dann verlöre jeder Filter
über die Dimension rund tausend Delikte, ohne zu scheitern. Sie nur als
`retired` zu markieren ist aber auch zu wenig: Das sagt, dass der Code weg ist,
nicht **wohin**. Wer eine Zeitreihe über mehrere Jahrgänge legt oder nach einem
Delikt filtert, übersieht sonst bei `26A` genau die Fälle, die früher `90A`
hiessen.

`dim_offense` trägt deshalb drei weitere Spalten: `resolution`, `replaced_by`
und `retirement_note`. Aufgelöst wird nur, wo es einen **Beleg** gibt — die
Spezifikation schweigt, sie kennt die Codes gar nicht mehr. Belegen lässt es
sich über die ILCS-Tabelle der Illinois State Police, dieselbe Quelle, die
schon die Chicagoer Gegenprobe trägt.

In [ ]:
if 'dim_offense' not in fehlend:
    print(f"{'Code':6}{'Bezeichnung':16}{'Aufloesung':18}{'heute':8}{'Faelle':>8}")
    for r in con.execute("""
        SELECT d.nibrs_code, d.nibrs_description, d.resolution,
               coalesce(d.replaced_by, '-') AS heute,
               count(f.offense_id) AS faelle
        FROM dim_offense d
        LEFT JOIN fact_offense f ON f.nibrs_code = d.nibrs_code
        WHERE d.retired GROUP BY 1, 2, 3, 4 ORDER BY 1""").fetchall():
        print(f'{r[0]:6}{r[1]:16}{r[2]:18}{r[3]:8}{r[4]:>8,}')

    print()
    print('Die Begruendungen:')
    for code, grund in con.execute(
        "SELECT nibrs_code, retirement_note FROM dim_offense "
        "WHERE retired ORDER BY nibrs_code"
    ).fetchall():
        print(f'  {code}: {grund}')

**`90A Bad Checks` löst sich auf `26A` auf**, und das ist belegt: Die
ILCS-Tabelle führt `720-5/17-1(B)` *Deceptive Practices – Bad Checks* unter
`26A False Pretenses/Swindle/Confidence Game`. Detroits MICR-Zeile `2606`
*Fraud – Bad Checks* liegt in derselben Deliktfamilie `26`.

**`90I Runaway` löst sich auf gar nichts auf** — und genau das ist die Antwort.
Weglaufen ist keine Straftat; deshalb hat das FBI den Code gestrichen. Die
ILCS-Tabelle kennt nur `720-5/10-6 Harboring a Runaway`, das *Beherbergen*, und
ordnet es `90Z` zu. Die 969 Zeilen bleiben zählbar, tragen aber kein heutiges
NIBRS-Delikt.

**`90E` und `90H` bleiben offen.** Öffentliche Trunkenheit ist in Illinois keine
Straftat, und die nächstliegenden Statute zum Spannertum
(`720-5/26-4 Unauthorized Video Recording`) verteilen sich je nach Ort auf `370`
und `90C` — kein einzelner Nachfolger. Beide kommen in den Falldaten 2024 nicht
vor, und **genau das erzwingt der Bau**: Ein Code mit
`resolution = 'unresolved'`, der in den Fakten auftaucht, lässt `reconcile`
fallen. Erscheint er in einem späteren Jahrgang, muss jemand einen Beleg
nachtragen, statt ihn stillschweigend mitlaufen zu lassen.

## 6. Grenzen

- **Keine Familienbezeichnungen.** `dim_offense` führt die Deliktfamilie als
  Kürzel (`13`, `23`, `26`), aber ohne Namen. Die NIBRS-Spezifikation nennt
  solche Namen nur verstreut im Fliesstext und führt keine auslesbare Liste;
  eine selbst geschriebene wäre eine gepflegte Übersetzung mehr — genau das,
  was mit `crime_category_mapping.csv` schon einmal entfernt wurde.
- **Ein Jahrgang.** Gold deckt 2024 ab. `dim_date` wird aus der Spanne der
  Fakten erzeugt und wächst mit.
- **Die Rate ist eine Näherung**, siehe Abschnitt 3: Mehrheitsregel,
  ACS-Schätzung, und rund zwei Prozent Randfälle.

In [ ]:
con.close()
print('Verbindung geschlossen.')